# Nanbeige4.2-3Bを無料版Google Colabで動かす

Nanbeige LLM Labの軽量Agentモデル **Nanbeige4.2-3B** を、Google Colab無料版T4 GPUで動かします。

基本構成は次の通りです。

```text
Google Colab Free / T4
        ↓
Nanbeige4.2-3B
        ↓
FP16でロード
        ↓
通常チャット
        ↓
Thinking ON / OFF
        ↓
簡単なTool Calling
        ↓
Gradio UI
```

公式weightsは約8.36GBです。まずはT4上でFP16ロードを試します。

> 公式モデルは最大256K contextをサポートしますが、無料版T4では短いcontextと小さな`max_new_tokens`から試します。
>
> また、公式の言語表記は英語・中国語です。日本語性能はこのNotebookで実験的に確認します。


In [1]:
# =========================================
# コード1 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

%pip -q install -U "transformers>=4.57,<5" accelerate sentencepiece

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory: %.2f GB" % (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
))


GPU 0: Tesla T4 (UUID: GPU-49c5373c-d716-743a-f379-e1b3ef08a586)
Python 3.12.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
# =========================================
# コード2 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
CACHE_DIR = PROJECT_DIR / "Program" / "hf_cache_nanbeige4_2_3b"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

usage = shutil.disk_usage(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("Drive free space: %.1f GB" % (usage.free / 1024**3))


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache_nanbeige4_2_3b
Drive free space: 14.9 GB


## モデルをFP16で読み込む

Nanbeige4.2-3Bの公式weightsは約8.36GBです。

まずは量子化せず、`FP16`で読み込みます。

T4ではBF16よりFP16の方が扱いやすいため、`torch.float16`を明示しています。

FP16ロードでOOMになった場合は、ランタイムを再起動してから最後の4bit fallbackセルを使ってください。


In [3]:
# =========================================
# コード3 Nanbeige4.2-3BをFP16で読み込む
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Nanbeige/Nanbeige4.2-3B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=False,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

model.eval()
INPUT_DEVICE = next(model.parameters()).device

print("model loaded:", MODEL_ID)
print("input device:", INPUT_DEVICE)
print("model dtype:", next(model.parameters()).dtype)

try:
    print("memory footprint: %.2f GB" % (
        model.get_memory_footprint() / 1024**3
    ))
except Exception:
    pass

print("GPU allocated: %.2f GB" % (
    torch.cuda.memory_allocated() / 1024**3
))
print("GPU reserved : %.2f GB" % (
    torch.cuda.memory_reserved() / 1024**3
))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

model loaded: Nanbeige/Nanbeige4.2-3B
input device: cuda:0
model dtype: torch.float16
memory footprint: 7.77 GB
GPU allocated: 7.78 GB
GPU reserved : 8.46 GB


In [6]:
# =========================================
# Nanbeige4.2-3B / Transformers互換性修正
# =========================================
from transformers.cache_utils import Cache

# Nanbeige4.2-3Bのcustom codeは旧API
# get_max_length() を利用している。
# 新しいTransformersではget_max_cache_shape()へ
# 変更されているため、互換aliasを追加する。
if not hasattr(Cache, "get_max_length"):

    def get_max_length_compat(self):
        return self.get_max_cache_shape()

    Cache.get_max_length = get_max_length_compat

print(
    "Cache.get_max_length:",
    hasattr(Cache, "get_max_length")
)

Cache.get_max_length: True


In [9]:
# =========================================
# コード4 生成関数
# Nanbeige4.2-3B compatibility version
# =========================================
import gc
import torch

@torch.inference_mode()
def nanbeige_generate(
    prompt,
    enable_thinking=False,
    max_new_tokens=256,
    temperature=0.6,
):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=enable_thinking,
        preserve_thinking=False,
    )

    input_ids = tokenizer(
        chat_prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(INPUT_DEVICE)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=int(max_new_tokens),

        # Nanbeige custom KV cacheと
        # 現行Transformersの互換性問題を回避
        use_cache=False,

        do_sample=True,
        temperature=float(temperature),
        top_p=0.95,
        top_k=20,

        # 公式READMEのgeneration例と同じEOS
        eos_token_id=166101,

        pad_token_id=(
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        ),
    )

    generated_ids = output_ids[
        0,
        input_ids.shape[-1]:
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    del input_ids
    del output_ids
    del generated_ids

    gc.collect()
    torch.cuda.empty_cache()

    return response

In [10]:
# =========================================
# コード5 日本語チャット
# =========================================
prompt = (
    "日本語で答えてください。\n\n"
    "生成AIにおけるハルシネーションとは何ですか？"
    "大学1年生にも分かるように簡潔に説明してください。"
)

answer = nanbeige_generate(
    prompt,
    enable_thinking=False,
    max_new_tokens=256,
)

print(answer)


## 生成AIにおける「ハルシネーション」とは？

### 簡潔な定義
**ハルシネーション（Hallucination）** とは、  
**生成AIが「存在しない事実や情報」を真実に見えるように「夢みたいに」作り出してしまうこと**のことです。  
つまり、**AIが自信満々に間違った答えを出す現象**です。

---

### 大学1年生でも理解できる例え
- 📚 **「知識を知らないと自信を持って答える」**  
  AIは「知識の海」を学習しましたが、不明な分野では、確率的に近い言葉を**つぎつぎ組み立てて**答えます。結果、実在しない文献・データ・歴史的事実を「真っ当に」述べてしまいます。
- 🎭 **「演技家のハルシネーション」**  
  AIが人間のように語りかけますが、


## Thinking ON / OFFを比較する

Nanbeige4.2-3Bのchat templateでは`enable_thinking`を切り替えられます。

通常チャットではThinkingをOFFにし、推論問題ではONにして比較します。

無料版T4では長すぎる生成を避けるため、Thinkingでも512 tokensまでに制限します。


In [11]:
# =========================================
# コード6 Thinking OFF
# =========================================
question = (
    "9.11と9.8では、どちらの数が大きいですか？"
    "理由も日本語で説明してください。"
)

print(
    nanbeige_generate(
        question,
        enable_thinking=False,
        max_new_tokens=256,
        temperature=0.6,
    )
)


**9.11と9.8では、9.8の方が大きい**です。以下に理由を日本語で説明します。

---

### 🔢 比較のポイント

1. **小数点以下の桁数を統一する**
   - 9.11 は「9」と「11/100」
   - 9.8 は「9」と「8/10」＝ **80/100**
   - 小数点以下の分母を100で表現すると、比較が明確になります。

2. **整数部分は同じ（9）**
   - 両者の整数部が「9」であるため、大小関係は**小数点以下**で決まります。

3. **小数点以下の比較**
   - 9.11 → 小数部分は **0.11**（11/100）
   - 9.8 → 小数部分は **0.80**（80/100）
   - 0.8


In [12]:
# =========================================
# コード7 Thinking ON
# =========================================
print(
    nanbeige_generate(
        question,
        enable_thinking=True,
        max_new_tokens=512,
        temperature=0.6,
    )
)


Okay， let's tackle this question: "9.11と9.8では、どちらの数が大きいですか？理由も日本語で説明してください。" First, I need to understand what the user is asking. They want to compare two numbers, 9.11 and 9.8, and determine which is larger, with reasons explained in Japanese.

Hmm, let me start by recalling how decimal numbers work. When comparing numbers with decimals, we look at the digits from left to right. Both numbers start with 9, so the whole number part is the same. The difference lies in the decimal parts: 0.11 versus 0.8. Wait, but 9.8 is the same as 9.80, right? Maybe aligning the decimal places would help. Let me write them out:

9.11
9.80 (since 9.8 is equivalent to 9.80)

Now, comparing digit by digit after the decimal point. The tenths place: for 9.11, the tenths digit is 1; for 9.80, it's 8. Since 8 is greater than 1 in the tenths place, 9.80 must be larger than 9.11. Therefore, 9.8 is bigger than 9.11.

Wait, but sometimes people get confused because 11 is larger than 8, but that's when comparing wh

## Tool Callingを試す

Nanbeige4.2-3Bはagentic / tool-useを重要な用途としており、chat templateへ`tools`を渡せます。

公式では`tool_call_format="xml"`が推奨されています。

ここでは外部APIを呼ばず、単純な掛け算ツールを提示し、モデルの生出力を確認します。


In [14]:
# =========================================
# コード8 Tool Callingの生出力を確認
# compatibility version
# =========================================
import torch
import gc

@torch.inference_mode()
def tool_call_test(
    user_prompt,
    max_new_tokens=512,
):
    tools = [
        {
            "type": "function",
            "function": {
                "name": "multiply",
                "description": "2つの数を掛け算します。",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {
                            "type": "number",
                            "description": "1つ目の数",
                        },
                        "b": {
                            "type": "number",
                            "description": "2つ目の数",
                        },
                    },
                    "required": ["a", "b"],
                },
            },
        }
    ]

    messages = [
        {
            "role": "user",
            "content": user_prompt,
        }
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=False,

        # Tool CallingではThinkingを利用
        enable_thinking=True,

        # 公式推奨
        preserve_thinking=True,
        tool_call_format="xml",
    )

    input_ids = tokenizer(
        chat_prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids.to(INPUT_DEVICE)

    outputs = model.generate(
        input_ids,

        max_new_tokens=max_new_tokens,

        # ★重要
        # Nanbeige custom KV cacheと
        # 現行Transformersの互換性問題を回避
        use_cache=False,

        do_sample=True,

        # 公式のAgent / Tool-use推奨値
        temperature=1.0,
        top_p=0.95,
        top_k=20,

        # Nanbeige公式Quickstartと同じEOS
        eos_token_id=166101,

        pad_token_id=(
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        ),
    )

    generated = outputs[
        0,
        input_ids.shape[-1]:
    ]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    del input_ids
    del outputs
    del generated

    gc.collect()
    torch.cuda.empty_cache()

    return response


tool_response = tool_call_test(
    "123.4 × 56.7を計算してください。"
    "必要なら利用可能なツールを使ってください。"
)

print(tool_response)

The user wants to calculate 123.4 × 56.7. I have a multiply tool available that can do this. Let me call it.
 </think> 

もちろん、利用可能なツール（掛け算）を使って計算します！

 <tool_call> 
<function=multiply>
<parameter=a>
123.4
</parameter>
<parameter=b>
56.7
</parameter>
</function>
 </tool_call>


In [17]:
# Gradioを介さず直接確認
test_result = gradio_generate(
    "日本語で答えてください。AIとは何ですか？",
    "Normal",
)

print(test_result)

**AI（人工知能）とは何か**について、日本語でご説明します。

---

### 📌 AI（人工知能）の定義
**AI（Artificial Intelligence：人工知能）**とは、**機械やプログラムが人間のような「知能」を模倣し、特定のタスクを行う技術**を指します。  
人間の知能（推論、学習、理解、判断など）の一部または全てをシステムやアルゴリズムで再現しようとする分野です。

---

### 🔍 AIの主な特徴・機能
- **データからの学習**：大量のデータを分析し、パターンや関係を学習します（例：画像認識、自然言語処理）。
- **推論・判断**：問題解決や予測を行います（例：医療診断支援、株式市場予測）。
- **言語処理**：日本語や他の言語の理解・生成を行うAI（例：


In [18]:
# =========================================
# コード8.5 RAM / VRAMを一部解放
# model と tokenizer は残す
# =========================================
import gc
import sys
import os
import psutil
import torch

def show_memory(label):
    process = psutil.Process(os.getpid())

    ram_gb = (
        process.memory_info().rss
        / 1024**3
    )

    print(f"\n=== {label} ===")
    print(
        f"Python RAM : {ram_gb:.2f} GB"
    )

    if torch.cuda.is_available():

        print(
            "GPU allocated: %.2f GB"
            % (
                torch.cuda.memory_allocated()
                / 1024**3
            )
        )

        print(
            "GPU reserved : %.2f GB"
            % (
                torch.cuda.memory_reserved()
                / 1024**3
            )
        )

        free, total = (
            torch.cuda.mem_get_info()
        )

        print(
            "GPU free     : %.2f GB"
            % (
                free / 1024**3
            )
        )


show_memory("before cleanup")


# -----------------------------
# 前のテストで作った大きめの変数を削除
# model/tokenizerは絶対に消さない
# -----------------------------
remove_names = [
    "tool_response",
    "outputs",
    "generated",
    "generated_ids",
    "output_ids",
    "inputs",
    "input_ids",
    "attention_mask",
    "chat_prompt",
    "answer",
]

for name in remove_names:
    if name in globals():
        del globals()[name]


# -----------------------------
# 直前の例外tracebackを解放
# -----------------------------
if getattr(
    sys,
    "last_traceback",
    None
) is not None:

    try:
        import traceback

        traceback.clear_frames(
            sys.last_traceback
        )

    except Exception:
        pass

sys.last_traceback = None
sys.last_value = None
sys.last_type = None


# -----------------------------
# IPythonのOut履歴を削除
# Namespace/modelは残す
# -----------------------------
try:
    ip = get_ipython()

    if ip is not None:
        ip.run_line_magic(
            "reset",
            "-f out"
        )

except Exception as e:
    print(
        "Output cache cleanup:",
        e
    )


# -----------------------------
# Python GC
# -----------------------------
gc.collect()


# -----------------------------
# CUDAの未使用cacheを解放
# -----------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()


show_memory("after cleanup")


print()
print(
    "model exists:",
    "model" in globals()
)

print(
    "tokenizer exists:",
    "tokenizer" in globals()
)


=== before cleanup ===
Python RAM : 2.14 GB
GPU allocated: 7.82 GB
GPU reserved : 8.46 GB
GPU free     : 5.95 GB
Flushing output cache (2 entries)

=== after cleanup ===
Python RAM : 2.14 GB
GPU allocated: 7.82 GB
GPU reserved : 8.46 GB
GPU free     : 5.95 GB

model exists: True
tokenizer exists: True


## Gradioで試す

通常回答とThinkingを切り替えられる簡単なUIを作ります。

無料版T4では、

```text
Normal   : 256 tokens
Thinking : 512 tokens
```

を上限にしています。


In [19]:
# =========================================
# コード9 Gradio UI
# Debug / Stable version
# =========================================
import gradio as gr
import gc
import torch
import traceback

print("Gradio:", gr.__version__)


def gradio_generate(prompt, mode):

    print("=== Gradio request started ===")
    print("mode:", mode)
    print("prompt:", prompt)

    if not (prompt or "").strip():
        raise gr.Error(
            "質問を入力してください。"
        )

    thinking = (
        mode == "Thinking"
    )

    # use_cache=Falseでは生成が遅いため
    # 最初は少なめにする
    max_tokens = (
        256
        if thinking
        else 128
    )

    try:

        result = nanbeige_generate(
            prompt,
            enable_thinking=thinking,
            max_new_tokens=max_tokens,
            temperature=0.6,
        )

        print("=== Generation completed ===")
        print(result)

        return result

    except Exception as e:

        print(
            "=== ERROR in gradio_generate ==="
        )

        # Colabセルへ完全なtracebackを出す
        traceback.print_exc()

        gc.collect()
        torch.cuda.empty_cache()

        # Gradio側にもエラーとして通知する
        raise gr.Error(
            f"{type(e).__name__}: {e}"
        ) from e


with gr.Blocks(
    title="Nanbeige4.2-3B"
) as demo:

    gr.Markdown(
        "## Nanbeige4.2-3B "
        "— Google Colab Free / T4"
    )

    prompt_box = gr.Textbox(
        label="Prompt",
        value=(
            "日本語で答えてください。\n\n"
            "生成AIのハルシネーションとは何ですか？"
        ),
        lines=5,
    )

    mode_box = gr.Radio(
        choices=[
            "Normal",
            "Thinking",
        ],
        value="Normal",
        label="Mode",
    )

    run_button = gr.Button(
        "Generate",
        variant="primary",
    )

    output_box = gr.Textbox(
        label="Answer",
        lines=12,
    )

    run_button.click(
        gradio_generate,
        inputs=[
            prompt_box,
            mode_box,
        ],
        outputs=output_box,

        # GPUモデルなので同時実行させない
        concurrency_limit=1,
    )


# GPU推論ではqueueを有効にする
demo.queue(
    default_concurrency_limit=1
)

print(
    "WARNING: share=Trueで"
    "一時公開URLが作成されます。"
)

demo.launch(
    share=True,
    inline=True,

    # Colab cellへtracebackを表示
    debug=False,

    # UI側にもエラーを表示
    show_error=False,
)

Gradio: 6.20.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f5f055e65b99d9cf9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
# =========================================
# コード10 GPUメモリ使用量
# =========================================
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU allocated: %.2f GB" % (
    torch.cuda.memory_allocated() / 1024**3
))
print("GPU reserved : %.2f GB" % (
    torch.cuda.memory_reserved() / 1024**3
))

free, total = torch.cuda.mem_get_info()

print("GPU free      : %.2f GB" % (free / 1024**3))
print("GPU total     : %.2f GB" % (total / 1024**3))


GPU allocated: 7.82 GB
GPU reserved : 8.46 GB
GPU free      : 5.95 GB
GPU total     : 14.56 GB


# 4bit fallback

**FP16ロードが無料版T4でOOMになった場合だけ使用してください。**

FP16ロードに失敗したあと、そのまま4bitセルを実行するとGPUメモリが残る場合があります。

必ずランタイムを再起動し、コード1・コード2を実行してから次のセルを使ってください。


In [ ]:
# =========================================
# オプション Nanbeige4.2-3BをNF4 4bitで読み込む
# =========================================
%pip -q install -U bitsandbytes

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "Nanbeige/Nanbeige4.2-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=False,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

model.eval()
INPUT_DEVICE = next(model.parameters()).device

print("model loaded:", MODEL_ID)
print("input device:", INPUT_DEVICE)
print("4bit:", getattr(model, "is_loaded_in_4bit", False))
print("GPU allocated: %.2f GB" % (
    torch.cuda.memory_allocated() / 1024**3
))
